In [83]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
from sklearn.datasets import make_classification
import torch

In [84]:
torch.manual_seed(42)

In [85]:
df = pd.read_csv("/fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [86]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [87]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [88]:
X_train = X_train/255.0
X_test = X_test/255.0

In [89]:
from torch.utils.data import Dataset, DataLoader

In [90]:
#create a CustomDataset class
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype = torch.long)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [91]:
#create train_dataset object
train_dataset = CustomDataset(X_train,y_train)

In [92]:
#create test_dataset object
test_dataset = CustomDataset(X_test,y_test)


In [93]:
#create train and test loader
train_loader = DataLoader(train_dataset, batch_size = 32 ,shuffle = True)
test_loader = DataLoader(train_dataset, batch_size = 32, shuffle =False) #shuffling may cause problem while testing and calculating accuracy


In [94]:
from torch.nn.modules.activation import ReLU
#define NN class
class MyNN(nn.Module):
  def __init__(self,num_features):

    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10),

    )
    ##we dont need to explicitly define softmax here as in pytorch in cross entropy loss has softmax implemented internally
  def forward(self,X):
    return self.model(X)

In [95]:
# set learning rate and epochs
learning_rate = 0.01
epochs = 100

In [96]:
#instatiate the model
import torch.optim as optim
mynn = MyNN(X_train.shape[1])


#loss function
loss_function = nn.CrossEntropyLoss()
#optimizer

optimizer = optim.SGD(mynn.parameters(), lr=learning_rate)


In [97]:
#training loop:
for epoch in range(epochs):
  total_epoch_loss= 0
  for batch_train , batch_label in train_loader:
    #forward pass
    y_pred = mynn(batch_train)


    #calculate loss
    loss = loss_function(y_pred ,batch_label)
    #back pass
    optimizer.zero_grad()
    loss.backward()


    #upgrade grades
    optimizer.step()
    total_epoch_loss += loss.item()
    avg_loss = total_epoch_loss / len(train_loader)

  print(f'epoch {epoch+1} : {avg_loss}')




epoch 1 : 2.2368171882629393
epoch 2 : 1.9528840390841167
epoch 3 : 1.4864923143386841
epoch 4 : 1.1698892108599346
epoch 5 : 0.9942735191186269
epoch 6 : 0.8960168067614237
epoch 7 : 0.8343262080351511
epoch 8 : 0.789007070461909
epoch 9 : 0.7555474547545115
epoch 10 : 0.7287200178702672
epoch 11 : 0.700073608358701
epoch 12 : 0.6826658533016841
epoch 13 : 0.6608437178532283
epoch 14 : 0.6402013136943181
epoch 15 : 0.6177361305554708
epoch 16 : 0.6091882650057475
epoch 17 : 0.5856802246967951
epoch 18 : 0.5769245492418608
epoch 19 : 0.564580753048261
epoch 20 : 0.5527976769208908
epoch 21 : 0.5441259907682737
epoch 22 : 0.5340593114495278
epoch 23 : 0.5224679175019264
epoch 24 : 0.5171100894610087
epoch 25 : 0.5065616236130397
epoch 26 : 0.5001970760027568
epoch 27 : 0.4924223287900289
epoch 28 : 0.48856514851252236
epoch 29 : 0.4775866487622261
epoch 30 : 0.4764286920428276
epoch 31 : 0.472826872865359
epoch 32 : 0.46381023039420444
epoch 33 : 0.45857009291648865
epoch 34 : 0.4533221

In [98]:
#set model to eval mode
mynn.eval()
#there are certain features which are different in eval and train, dropout, mean-std calculation

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [99]:
#evaluation code
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    outputs = mynn(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.9225
